# 10. Galutinis G5 TEST (vieną kartą)

G5 (`γ=0.5`, `λ=0.1`, `n_tree=100`, `mtry=√p`) pasirinktas **tik** 09 etapo TRAIN GroupKFold. Šiame notebook TEST naudojamas **vieną kartą** po TRAIN fit. Parametrų po TEST nekeičiame.

Metodas: **GRRF-inspired regularized feature selection** + `StandardScaler` + `LinearSVC(C=0.01)`.

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
GAMMA = 0.5
LAMBDA = 0.1
N_TREE = 100
MTRY = "sqrt"

BASE_F1 = 0.897129864866
BASE_BA = 0.893665761340
BASE_ACC = 0.890016
BASE_ERR = 347

np.random.seed(RANDOM_STATE)
print("G5 parametrai fiksuoti iš TRAIN CV. Po TEST jie nebus keičiami.")
print(f"gamma={GAMMA}, lambda={LAMBDA}, n_tree={N_TREE}, mtry={MTRY}")

G5 parametrai fiksuoti iš TRAIN CV. Po TEST jie nebus keičiami.
gamma=0.5, lambda=0.1, n_tree=100, mtry=sqrt


In [2]:
cwd = Path.cwd().resolve()
project_root = None
for cand in [cwd, cwd.parent]:
    if (cand / "results").exists() and (cand / "notebooks").exists():
        project_root = cand
        break
if project_root is None:
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

results_dir = project_root / "results"
split = np.load(results_dir / "subject_disjoint_split.npz", allow_pickle=True)
X_train = np.asarray(split["X_train_final"], dtype=np.float64)
X_test = np.asarray(split["X_test_final"], dtype=np.float64)
y_train = np.asarray(split["y_train_final"], dtype=np.int64)
y_test = np.asarray(split["y_test_final"], dtype=np.int64)
subjects_train = np.asarray(split["subjects_train_final"], dtype=np.int64)
subjects_test = np.asarray(split["subjects_test_final"], dtype=np.int64)
feature_names = np.array([str(n) for n in split["feature_names"]])
activity_ids = np.asarray(split["activity_ids"], dtype=np.int64)
activity_names = np.array([str(n) for n in split["activity_names"]])
id_to_name = {int(i): str(n) for i, n in zip(activity_ids, activity_names)}

train_ids = sorted(set(subjects_train.tolist()))
test_ids = sorted(set(subjects_test.tolist()))
overlap = sorted(set(train_ids) & set(test_ids))
print("TRAIN", X_train.shape, "people", train_ids)
print("TEST ", X_test.shape, "people", test_ids)
print("intersection", overlap)
assert overlap == []
assert X_train.shape == (7144, 561) and X_test.shape == (3155, 561)
print("G5 pasirinkimas: 09_GRRF_inspired_cv.ipynb, be TEST metrikų.")

TRAIN (7144, 561) people [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15, 17, 19, 20, 21, 22, 23, 26, 27, 30]
TEST  (3155, 561) people [9, 10, 13, 16, 18, 24, 25, 28, 29]
intersection []
G5 pasirinkimas: 09_GRRF_inspired_cv.ipynb, be TEST metrikų.


In [3]:
def gini_parent(counts):
    n = counts.sum()
    if n <= 0:
        return 0.0
    return 1.0 - float(np.dot(counts, counts)) / float(n * n)


def best_threshold_and_gain(x_col, y_node, n_classes, min_leaf):
    n = x_col.shape[0]
    if n < 2 * min_leaf:
        return -1.0, np.nan
    order = np.argsort(x_col, kind="mergesort")
    x_s = x_col[order]
    y_s = y_node[order]
    if x_s[0] == x_s[-1]:
        return -1.0, np.nan
    oh = np.zeros((n, n_classes), dtype=np.float64)
    oh[np.arange(n), y_s] = 1.0
    left = np.cumsum(oh, axis=0)
    total = left[-1]
    g_parent = gini_parent(total)
    i = np.arange(min_leaf, n - min_leaf + 1)
    valid = x_s[i] != x_s[i - 1]
    if not np.any(valid):
        return -1.0, np.nan
    left_c = left[i - 1]
    left_n = i.astype(np.float64)
    right_c = total - left_c
    right_n = float(n) - left_n
    g_left = 1.0 - np.sum(left_c * left_c, axis=1) / (left_n * left_n)
    g_right = 1.0 - np.sum(right_c * right_c, axis=1) / (right_n * right_n)
    gain = g_parent - (left_n / float(n)) * g_left - (right_n / float(n)) * g_right
    gain = np.where(valid, gain, -np.inf)
    j = int(np.argmax(gain))
    thr = 0.5 * (x_s[i[j]] + x_s[i[j] - 1])
    return float(gain[j]), float(thr)


def grow_regularized_tree(X, y, rng, alpha, max_features, min_samples_split, min_samples_leaf):
    n_samples, n_features = X.shape
    n_classes = int(y.max()) + 1
    used = set()
    used_in_tree = set()

    def majority(idx):
        return int(np.argmax(np.bincount(y[idx], minlength=n_classes)))

    def recurse(idx):
        y_n = y[idx]
        if idx.size < min_samples_split or np.unique(y_n).size == 1:
            return {"leaf": True, "pred": majority(idx)}
        mtry = min(max_features, n_features)
        cand = rng.choice(n_features, size=mtry, replace=False)
        best_gain, best_k, best_thr = -1.0, None, None
        x_node = X[idx]
        for k in cand:
            raw_gain, thr = best_threshold_and_gain(x_node[:, k], y_n, n_classes, min_samples_leaf)
            if raw_gain < 0 or not np.isfinite(thr):
                continue
            gain = raw_gain if k in used else raw_gain * float(alpha[k])
            if gain > best_gain:
                best_gain, best_k, best_thr = gain, int(k), thr
        if best_k is None or best_gain <= 0:
            return {"leaf": True, "pred": majority(idx)}
        used.add(best_k)
        used_in_tree.add(best_k)
        left_mask = x_node[:, best_k] <= best_thr
        left_idx, right_idx = idx[left_mask], idx[~left_mask]
        if left_idx.size == 0 or right_idx.size == 0:
            return {"leaf": True, "pred": majority(idx)}
        return {
            "leaf": False,
            "feature": best_k,
            "left": recurse(left_idx),
            "right": recurse(right_idx),
        }

    recurse(np.arange(n_samples, dtype=np.int64))
    return used_in_tree


def grrf_inspired_select_train(X_fit, y_fit):
    """Galutinė G5 kaukė: tik visas TRAIN. (5) RF importance, (8) alpha, (6)–(7) medžiai."""
    n_features = X_fit.shape[1]
    mtry = max(1, int(np.sqrt(n_features)))
    rf = RandomForestClassifier(
        n_estimators=N_TREE,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_fit, y_fit)
    imp = rf.feature_importances_.astype(np.float64)
    imp_norm = imp / float(imp.max())
    alpha = (1.0 - GAMMA) * LAMBDA + GAMMA * imp_norm
    alpha = np.clip(alpha, 0.0, 1.0)
    _, y_enc = np.unique(y_fit, return_inverse=True)
    rng = np.random.RandomState(RANDOM_STATE)
    selected = set()
    t0 = time.perf_counter()
    for t in range(N_TREE):
        boot = rng.randint(0, X_fit.shape[0], size=X_fit.shape[0])
        selected |= grow_regularized_tree(
            X_fit[boot], y_enc[boot], rng, alpha, mtry, 2, 1
        )
        if (t + 1) % 25 == 0:
            print(f"  medis {t+1}/{N_TREE}, n_feat={len(selected)}, {time.perf_counter()-t0:.1f}s")
    idx = np.array(sorted(selected), dtype=np.int64)
    return idx, imp, imp_norm, alpha, rf, mtry

## Feature mask tik iš viso TRAIN

`X_test` čia nepaduodamas.

In [4]:
print("GRRF-inspired G5: fit tik X_train, y_train")
sel, imp, imp_norm, alpha, rf_imp, mtry = grrf_inspired_select_train(X_train, y_train)
n_sel = int(sel.size)
n_rem = 561 - n_sel
pct_rem = 100.0 * n_rem / 561.0
print(f"mtry={mtry} (sqrt(561))")
print(f"Atrinkta: {n_sel} / 561")
print(f"Pašalinta: {n_rem} ({pct_rem:.2f} %)")
print("RF importance n_features_in_=", rf_imp.n_features_in_, "n_samples=", X_train.shape[0])

GRRF-inspired G5: fit tik X_train, y_train


  medis 25/100, n_feat=371, 23.5s


  medis 50/100, n_feat=451, 46.8s


  medis 75/100, n_feat=496, 71.1s


  medis 100/100, n_feat=523, 97.0s
mtry=23 (sqrt(561))
Atrinkta: 523 / 561
Pašalinta: 38 (6.77 %)
RF importance n_features_in_= 561 n_samples= 7144


In [5]:
def make_svm():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LinearSVC(
                    C=0.01,
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    dual="auto",
                ),
            ),
        ]
    )


pipe = make_svm()
pipe.fit(X_train[:, sel], y_train)
print("Scaler fit TRAIN n_features_in_=", pipe.named_steps["scaler"].n_features_in_)
print("SVM fit TRAIN classes_=", pipe.named_steps["clf"].classes_)

pred = pipe.predict(X_test[:, sel])
g5_f1 = float(f1_score(y_test, pred, average="macro"))
g5_ba = float(balanced_accuracy_score(y_test, pred))
g5_acc = float(accuracy_score(y_test, pred))
g5_err = int(np.sum(pred != y_test))

d_f1 = g5_f1 - BASE_F1
d_ba = g5_ba - BASE_BA
d_acc = g5_acc - BASE_ACC
d_err = g5_err - BASE_ERR

labels = activity_ids.tolist()
names = [id_to_name[int(i)] for i in labels]
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=names, columns=names)

print("G5 TEST Macro-F1", g5_f1)
print("G5 TEST BA", g5_ba)
print("G5 TEST Acc", g5_acc)
print("G5 TEST errors", g5_err)
print(cm_df.to_string())
print("Δ F1", d_f1, "Δ BA", d_ba, "Δ Acc", d_acc, "Δ err", d_err)
print("features reduced %", pct_rem)

off = []
for i, ti in enumerate(names):
    for j, pj in enumerate(names):
        if i != j and cm[i, j] > 0:
            off.append((int(cm[i, j]), ti, pj))
off.sort(reverse=True)
print("Didžiausios klaidos (už įstrižainės):")
for n, a, b in off[:8]:
    print(f"  {a} → {b}: {n}")

Scaler fit TRAIN n_features_in_= 523
SVM fit TRAIN classes_= [1 2 3 4 5 6]
G5 TEST Macro-F1 0.8940470852363642
G5 TEST BA 0.889971746112964
G5 TEST Acc 0.8865293185419968
G5 TEST errors 358
                    WALKING  WALKING_UPSTAIRS  WALKING_DOWNSTAIRS  SITTING  STANDING  LAYING
WALKING                 504                 2                   0        2         0       0
WALKING_UPSTAIRS          5               414                   1       64         0       0
WALKING_DOWNSTAIRS        3                21                 409        3         0       0
SITTING                   0                 1                   0      470        73       0
STANDING                  0                 0                   0       98       486       0
LAYING                    0                 0                   0        0        85     514
Δ F1 -0.003082779629635768 Δ BA -0.0036940152270359405 Δ Acc -0.003486681458003238 Δ err 11
features reduced % 6.7736185383244205
Didžiausios klaidos (už įstri

In [6]:
cmp = pd.DataFrame(
    [
        {
            "Model": "SVM (all 561 features)",
            "N features": 561,
            "Macro-F1": BASE_F1,
            "Balanced Accuracy": BASE_BA,
            "Accuracy": BASE_ACC,
            "TEST Errors": BASE_ERR,
        },
        {
            "Model": "G5 GRRF-inspired + SVM",
            "N features": n_sel,
            "Macro-F1": g5_f1,
            "Balanced Accuracy": g5_ba,
            "Accuracy": g5_acc,
            "TEST Errors": g5_err,
        },
    ]
)
cmp["Macro-F1 Δ vs A"] = cmp["Macro-F1"] - BASE_F1
cmp["BA Δ vs A"] = cmp["Balanced Accuracy"] - BASE_BA
cmp["Acc Δ vs A"] = cmp["Accuracy"] - BASE_ACC
cmp["Errors Δ vs A"] = cmp["TEST Errors"] - BASE_ERR
print(cmp.to_string(index=False))

feat_df = pd.DataFrame(
    {
        "feature_index": sel,
        "feature_name": feature_names[sel],
        "rf_importance_train_only": imp[sel],
        "importance_normalized_eq8": imp_norm[sel],
        "alpha_k_eq8": alpha[sel],
    }
)
feat_path = results_dir / "grrf_g5_selected_features.csv"
cmp_path = results_dir / "grrf_g5_vs_svm_test.csv"
cm_path = results_dir / "grrf_g5_svm_confusion_matrix.csv"
feat_df.to_csv(feat_path, index=False)
cmp.to_csv(cmp_path, index=False)
cm_out = cm_df.copy()
cm_out.insert(0, "true_activity", names)
cm_out.to_csv(cm_path, index=False)
print("Išsaugota:", feat_path)
print("Išsaugota:", cmp_path)
print("Išsaugota:", cm_path)
print("08/09 CSV neperrašyti.")

small = abs(d_f1) < 0.01 and abs(d_ba) < 0.01
print("Interpretacija:")
if small and d_err <= 15:
    print("Skirtumas mažas; nevadiname reikšmingu pagerėjimu ar pablogėjimu.")
elif d_f1 <= -0.01:
    print("G5 TEST Macro-F1 žemesnis nei 561 SVM; kokybė ne visai išlaikyta.")
elif d_f1 >= 0.01:
    print("G5 TEST Macro-F1 aukštesnis, bet vis tiek nedeklaruojame didelio laimėjimo be daugiau testų.")
else:
    print("Žr. deltas; TEST skirtumas nedidelis palyginti su 08 G2 (−0.016 F1).")

                 Model  N features  Macro-F1  Balanced Accuracy  Accuracy  TEST Errors  Macro-F1 Δ vs A  BA Δ vs A  Acc Δ vs A  Errors Δ vs A
SVM (all 561 features)         561  0.897130           0.893666  0.890016          347         0.000000   0.000000    0.000000              0
G5 GRRF-inspired + SVM         523  0.894047           0.889972  0.886529          358        -0.003083  -0.003694   -0.003487             11
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_g5_selected_features.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_g5_vs_svm_test.csv
Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_g5_svm_confusion_matrix.csv
08/09 CSV neperrašyti.
Interpretacija:
Skirtumas mažas; nevadiname reikšmingu pagerėjimu

In [7]:
print("=== GALUTINIS DATA LEAKAGE AUDITAS ===")
print("1) TRAIN∩TEST subject:", overlap, "tuščia=", overlap == [])
print("2) G5 pasirinktas 09 TRAIN GroupKFold, ne TEST.")
print("3) Mask: RF+GRRF-inspired tik X_train", X_train.shape, "sel", n_sel)
print("4) Scaler n_features_in_", pipe.named_steps["scaler"].n_features_in_, "== n_sel", n_sel == pipe.named_steps["scaler"].n_features_in_)
print("5) SVM fit TRAIN, C=0.01, TEST tik predict.")
print("6) Po šio TEST parametrų nekeičiame.")

=== GALUTINIS DATA LEAKAGE AUDITAS ===
1) TRAIN∩TEST subject: [] tuščia= True
2) G5 pasirinktas 09 TRAIN GroupKFold, ne TEST.
3) Mask: RF+GRRF-inspired tik X_train (7144, 561) sel 523
4) Scaler n_features_in_ 523 == n_sel True
5) SVM fit TRAIN, C=0.01, TEST tik predict.
6) Po šio TEST parametrų nekeičiame.
